In [17]:
import pandas as pd
DATA_PATH = "data/amazon_reviews_us_Digital_Video_Games_v1_00.tsv.zip"

df = pd.read_csv(DATA_PATH, sep="\t")
print("Default settings:", df.shape)

Default settings: (144724, 15)


In [ ]:
#144,724 rows, 15 columns.

In [2]:
df = pd.read_csv(DATA_PATH, sep="\t", quoting=3)
print("quoting=3:", df.shape)
print(df.dtypes)

quoting=3: (145431, 15)
marketplace            str
customer_id          int64
review_id              str
product_id             str
product_parent       int64
product_title          str
product_category       str
star_rating          int64
helpful_votes        int64
total_votes          int64
vine                   str
verified_purchase      str
review_headline        str
review_body            str
review_date            str
dtype: object


In [ ]:
#default loaded 144,724 rows and quoting=3 loaded 145,431, so 707 rows went missing. pandas did not warn me about it.
#a load finishing with no errors doesn't mean it's right. should check the row count against something else (line count of the file, the docs) before using the data.

In [3]:
pd.set_option("display.max_colwidth", 400)

# Pick a random sample of five that shows the typical problems in this data
for seed in range(200):
    five = df.sample(5, random_state=seed)
    if five["review_body"].str.contains("<br />|&#", na=False).sum() >= 3:
        break
print("seed:", seed)
for i, row in five.iterrows():
    print(i, "| stars:", row["star_rating"], "|", row["product_title"])
    print(str(row["review_body"])[:350])
    print("-" * 60)

seed: 14
106923 | stars: 2 | SimCity - Limited Edition
Google \\"SimCity 2013: The deception continues ...\\"<br /><br />- The presence of empty spaces not editable (useless empty spaces) with automatic road connections.<br />- Absence of terraforming.<br />- Absence of farms.<br />- Not being able to save your maps on the hard drive (you will not be able to reload the city to a stage prior to disaster
------------------------------------------------------------
118701 | stars: 4 | ANNO 2070 [Download]
Easy enough to jump into and can become difficulty as game continues. Though the side quests become tedious after a while, the game has decent replay ability. I still do not think I like it enough to purchase the (somewhat pricey) add-on (deepwater).
------------------------------------------------------------
35099 | stars: 4 | Valkyria Chronicles
Finally, a port of the PS3 game to the PC.  Combines turn based strategy with shooting/action elements, set in an alternative &#34;Europe&#34

In [ ]:
#weird stuff: <br /> tags, &#34; instead of quote marks, \" escaped quotes, some reviews are just one word.
#probably from scraping amazon's html pages. the text was stored as html and nobody cleaned it.

In [4]:
df["star_rating"] = pd.to_numeric(df["star_rating"], errors="coerce")
print(df["star_rating"].isna().sum(), "rows with non-numeric star rating")
print(df["star_rating"].value_counts().sort_index())
print(df["star_rating"].value_counts(normalize=True).sort_index().round(3))

0 rows with non-numeric star rating
star_rating
1    24970
2     7749
3    11629
4    20406
5    80677
Name: count, dtype: int64
star_rating
1    0.172
2    0.053
3    0.080
4    0.140
5    0.555
Name: proportion, dtype: float64


In [ ]:
#5 stars is 55%, 1 star is 17%, 4 is 14%, 3 is 8%, 2 is 5%. a model that says positive for everything would already look accurate, so accuracy can be misleading.

In [5]:
df["product_title"].value_counts().head(10)

product_title
Playstation Network Card                                          13642
Xbox Live Subscription                                             7307
Playstation Plus Subscription                                      4712
Xbox Live Gift Card                                                3438
SimCity - Limited Edition                                          3421
Xbox 360 Live Points Card                                          3036
Battlefield 4                                                      1453
Final Fantasy XIV: A Realm Reborn                                  1273
Battlefield 3                                                      1160
Command and Conquer The Ultimate Collection [Online Game Code]      875
Name: count, dtype: int64

In [ ]:
#most reviewed "games" are gift cards and subscriptions (PSN card 13,642, Xbox Live 7,307). those reviews are about codes, not gameplay.

In [6]:
print("Reviews with <br /> before cleaning:", df["review_body"].str.contains("<br />", na=False).sum())
print("Empty/missing reviews:", df["review_body"].isna().sum() + (df["review_body"].fillna("").str.strip() == "").sum())

df = df.dropna(subset=["star_rating"])
df = df[df["review_body"].notna() & (df["review_body"].str.strip() != "")]
df["review_body"] = df["review_body"].str.replace("<br />", " ", regex=False)
df["star_rating"] = df["star_rating"].astype(int)

print("Reviews with <br /> after cleaning:", df["review_body"].str.contains("<br />").sum())
print("Rows left:", len(df))

for i in five.index.intersection(df.index):
    print(i, "| stars:", df.loc[i, "star_rating"], "|", df.loc[i, "product_title"])
    print(df.loc[i, "review_body"][:350])
    print("-" * 60)

Reviews with <br /> before cleaning: 25699
Empty/missing reviews: 6
Reviews with <br /> after cleaning: 0
Rows left: 145428
106923 | stars: 2 | SimCity - Limited Edition
Google \\"SimCity 2013: The deception continues ...\\"  - The presence of empty spaces not editable (useless empty spaces) with automatic road connections. - Absence of terraforming. - Absence of farms. - Not being able to save your maps on the hard drive (you will not be able to reload the city to a stage prior to disasters). - Road connections pl
------------------------------------------------------------
118701 | stars: 4 | ANNO 2070 [Download]
Easy enough to jump into and can become difficulty as game continues. Though the side quests become tedious after a while, the game has decent replay ability. I still do not think I like it enough to purchase the (somewhat pricey) add-on (deepwater).
------------------------------------------------------------
35099 | stars: 4 | Valkyria Chronicles
Finally, a port of the PS3

In [ ]:
#25,699 reviews had <br /> before cleaning, 0 after. also dropped 6 empty reviews.
#still see &#34; and \" in them. probably doesn't change positive vs negative much, just extra junk tokens.

In [7]:
sample = pd.concat([df[df["star_rating"] == s].sample(100, random_state=42) for s in range(1, 6)]).reset_index(drop=True)
extra_one_star = df[(df["star_rating"] == 1) & ~df["review_id"].isin(sample["review_id"])].sample(20, random_state=42).reset_index(drop=True)
print(sample["star_rating"].value_counts().sort_index())
print(len(extra_one_star), "extra one-star reviews")

star_rating
1    100
2    100
3    100
4    100
5    100
Name: count, dtype: int64
20 extra one-star reviews


In [ ]:
#same number per star because 5 stars is 55% of the data. a random 500 would only have ~27 two star and ~40 three star reviews, too few to measure anything.

In [8]:
from transformers import pipeline

sentiment = pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")
one = sample.loc[0]
print("Stars:", one["star_rating"])
print(one["review_body"][:500])
sentiment(one["review_body"], truncation=True)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Stars: 1
Terrible, confusing interface. Looks like a cartoon. Didn't really like the long, drawn out stories in my player. Game play and graphics weren't that much improved $30 extra we had to pay (Now we get VC yea! not).


[{'label': 'NEGATIVE', 'score': 0.9998071789741516}]

In [ ]:
#keys are label and score. score is how confident the model is in that label (0 to 1). 0.9998 NEGATIVE here.
#yes it agrees with the 1 star rating.

In [9]:
import time

texts = sample["review_body"].tolist()
start = time.time()
results = sentiment(texts, truncation=True, batch_size=16)
elapsed = time.time() - start
sample["sentiment"] = [r["label"] for r in results]
sample["score"] = [r["score"] for r in results]
print(f"{elapsed:.1f} seconds for {len(texts)} reviews")
print(f"Estimated for {len(df):,} reviews: {elapsed / len(texts) * len(df) / 3600:.2f} hours")

2.4 seconds for 500 reviews
Estimated for 145,428 reviews: 0.20 hours


In [ ]:
#500 reviews took about 3.2 seconds. all 145k would be about 0.26 hours (~15 min) at that speed. faster than expected, probably my machine.
#truncation=True cuts long reviews to the model's 512 token limit so they don't error out.

In [10]:
key = sample[sample["star_rating"] != 3].copy()
key["truth"] = key["star_rating"].apply(lambda s: "POSITIVE" if s >= 4 else "NEGATIVE")
key["correct"] = key["truth"] == key["sentiment"]
print("Accuracy:", round(key["correct"].mean(), 3), f"({key['correct'].sum()}/{len(key)})")

Accuracy: 0.875 (350/400)


In [ ]:
#left out 3 stars because they aren't clearly positive or negative, so there's no right answer in the key.
#accuracy is 87.5% (350/400). decent, but i wouldn't trust it alone in a product. wrong about 1 in 8.

In [11]:
print(key.groupby("star_rating")["correct"].mean().round(3))
print()
print("What the model said about 3-star reviews:")
print(sample[sample["star_rating"] == 3]["sentiment"].value_counts())

star_rating
1    0.96
2    0.91
4    0.80
5    0.83
Name: correct, dtype: float64

What the model said about 3-star reviews:
sentiment
NEGATIVE    59
POSITIVE    41
Name: count, dtype: int64


In [ ]:
#4 stars is the worst (80% right), then 5 stars (83%).
#for 3 stars the model had to pick a side: 59 NEGATIVE, 41 POSITIVE. there's no neutral option.

In [12]:
mistakes = key[~key["correct"]].sample(5, random_state=1)
for i, row in mistakes.iterrows():
    print(f"Stars: {row['star_rating']} | model: {row['sentiment']} ({row['score']:.2f})")
    print(row["review_body"][:600])
    print("-" * 60)

Stars: 4 | model: NEGATIVE (0.98)
Before you complain install xlive to get the game to launch
------------------------------------------------------------
Stars: 5 | model: NEGATIVE (0.98)
This game is just like the origional PS1/PC copy. They did touch it up with smoothing out the graphics a little but didnt upgrade them. They did add more enemies and side quests. They also made it compatible with all the newer computers/windows based tablets (with keyboard but controller is recommended) I played this on my windows surface pro using a usb controller (as well as my desktop too). If you are wanting an easier way to play this game use either a logitec controller (like the Playstation style controllers) or a 360 usb controller.
------------------------------------------------------------
Stars: 5 | model: NEGATIVE (0.99)
It's Titanfall. Either you hate it or you love it! :D  In a nutshell, its a beefed up arcade game. near non-existent story line, weak campaign, and no named characters or

In [ ]:
#1 (4 stars, xlive): sounds like a complaint but the customer was giving advice. model's fault.
#2 (5 stars, PS1 port): mixed review, lists what wasn't upgraded then recommends it. hard case.
#3 (5 stars, titanfall): lots of criticism then "great game". rating and text disagree, so the rating is kind of misleading.
#4 (5 stars, military gift card): complaints are about the old way of buying cards, not the product. model's fault.
#5 (1 star, wasted $7): "im sure its a good game" pulled it positive. model's fault, only 0.73 confident.

In [13]:
stars_model = pipeline("text-classification", model="nlptown/bert-base-multilingual-uncased-sentiment")
star_results = stars_model(sample["review_body"].tolist(), truncation=True, batch_size=16)
sample["pred_stars"] = [int(r["label"][0]) for r in star_results]
exact = (sample["pred_stars"] == sample["star_rating"]).mean()
within1 = ((sample["pred_stars"] - sample["star_rating"]).abs() <= 1).mean()
print(f"Exact: {exact:.3f} | Within one star: {within1:.3f}")
print(pd.crosstab(sample["star_rating"], sample["pred_stars"]))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Exact: 0.574 | Within one star: 0.916
pred_stars    1   2   3   4   5
star_rating                    
1            71  19   6   3   1
2            28  52  14   6   0
3            10  20  51  14   5
4             0   0  25  52  23
5             2   4   5  28  61


In [ ]:
#exact 57.4%, within one star 91.6%. most misses are the neighboring rating.
#i'd use the stars model for the dashboard. it gives a scale and is within one star ~92% of the time, so errors average out over lots of reviews. the other model only says good or bad.

In [14]:
zero_shot = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
labels = ["shipping or delivery", "product quality or bugs", "price or value", "customer service", "technical problems or compatibility"]
zs_texts = extra_one_star["review_body"].tolist()

start = time.time()
zs_results = zero_shot(zs_texts, candidate_labels=labels, truncation=True)
print(f"{time.time() - start:.0f} seconds")
extra_one_star["topic"] = [r["labels"][0] for r in zs_results]
extra_one_star["topic_score"] = [r["scores"][0] for r in zs_results]
print(extra_one_star["topic"].value_counts())

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

3 seconds
topic
technical problems or compatibility    9
price or value                         5
product quality or bugs                4
shipping or delivery                   1
customer service                       1
Name: count, dtype: int64


In [ ]:
#technical problems/compatibility is the top (9 of 20), then price/value (5), product quality (4), shipping (1), customer service (1).

In [15]:
for i in range(3):
    print("Topic:", extra_one_star.loc[i, "topic"], f"({extra_one_star.loc[i, 'topic_score']:.2f})")
    print(extra_one_star.loc[i, "review_body"][:500])
    print("-" * 60)

Topic: shipping or delivery (0.47)
Didn't order it
------------------------------------------------------------
Topic: technical problems or compatibility (0.59)
We never got this program to run properly!  Have returned it for a full refund without any problems.  The online game site is very tricky to use.  Would not use it again.
------------------------------------------------------------
Topic: price or value (0.40)
Never got the code, was still charged
------------------------------------------------------------


In [ ]:
#mostly makes sense. "never got the code, was still charged" got price or value (0.40) but it's really delivery/billing. "didn't order it" got shipping (0.47), probably an unauthorized purchase. low scores show the model wasn't sure.

In [16]:
labels2 = labels + ["defective product", "weather"]
zs_results2 = zero_shot(zs_texts[:5], candidate_labels=labels2, truncation=True)
for i in range(5):
    print(f"Review {i}")
    print("  original:", [(l, round(s, 2)) for l, s in zip(zs_results[i]["labels"][:3], zs_results[i]["scores"][:3])])
    print("  new     :", [(l, round(s, 2)) for l, s in zip(zs_results2[i]["labels"][:3], zs_results2[i]["scores"][:3])])
    print("  sum of new scores:", round(sum(zs_results2[i]["scores"]), 3))

Review 0
  original: [('shipping or delivery', 0.47), ('price or value', 0.18), ('product quality or bugs', 0.16)]
  new     : [('shipping or delivery', 0.35), ('defective product', 0.24), ('price or value', 0.13)]
  sum of new scores: 1.0
Review 1
  original: [('technical problems or compatibility', 0.59), ('product quality or bugs', 0.33), ('customer service', 0.05)]
  new     : [('defective product', 0.51), ('technical problems or compatibility', 0.28), ('product quality or bugs', 0.16)]
  sum of new scores: 1.0
Review 2
  original: [('price or value', 0.4), ('technical problems or compatibility', 0.34), ('shipping or delivery', 0.13)]
  new     : [('defective product', 0.29), ('price or value', 0.28), ('technical problems or compatibility', 0.24)]
  sum of new scores: 1.0
Review 3
  original: [('price or value', 0.53), ('product quality or bugs', 0.2), ('customer service', 0.1)]
  new     : [('price or value', 0.42), ('product quality or bugs', 0.16), ('weather', 0.11)]
  sum of ne

In [ ]:
#added "defective product" (overlaps product quality) and "weather" (irrelevant). defective product took the top spot in some reviews (review 1: 0.51, technical problems fell from 0.59 to 0.28) and shipping fell from 0.47 to 0.35 in review 0. weather still got 0.11 on review 3.
#scores add up to 1, so the labels split the same probability. adding a label takes share from the others, even a useless one.
#lesson: labels shouldn't overlap, should cover real complaints (mine had no billing/code delivery), and need testing on real reviews first.